<a href="https://colab.research.google.com/github/hoshi-3104-com/california-house-price-competition/blob/main/notebooks/04_house_price_model_comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# driveのマウント
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Kaggle APIを利用できるようにするため、kaggle.jsonからusernameとkeyを設定する
import os
import json
f = open("/content/drive/MyDrive/house_price_competition_hoshino/kaggle.json", 'r') # ディレクトリは必要に応じて変更してください
json_data = json.load(f)
os.environ['KAGGLE_USERNAME'] = json_data['username']
os.environ['KAGGLE_KEY'] = json_data['key']

In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns

# 前処理(正規化・標準化)
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# データ分割
from sklearn.model_selection import train_test_split

# 線形モデル
from sklearn.linear_model import LinearRegression

# 精度評価
from sklearn.metrics import mean_squared_error

# グラフをアウトプット行に出力するためのマジックコマンド
%matplotlib inline

In [ ]:
train = pd.read_csv('/content/drive/MyDrive/house_price_competition_hoshino/data/train_data_raw.csv')
test = pd.read_csv('/content/drive/MyDrive/house_price_competition_hoshino/data/test_data_raw.csv')
sample = pd.read_csv('/content/drive/MyDrive/house_price_competition_hoshino/data/sample_raw.csv')

# 2. データの切り分け
X_all_train = train.drop(["Price", "id"], axis=1)
y_all_train = train["Price"]
X_test = test.drop(["id"], axis=1) if "id" in test.columns else test

In [ ]:
# 必要なライブラリの追加インポート
from sklearn.model_selection import KFold, cross_val_score
from sklearn.feature_selection import RFE
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# 1. 検証モデルの定義（似たモデルを近くに配置 / 各パラメータはデフォルト）
models = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(),
    "Lasso": Lasso(),
    "ElasticNet": ElasticNet(), # 追加モデル1
    "RandomForest": RandomForestRegressor(random_state=42),
    "GradientBoosting": GradientBoostingRegressor(random_state=42), # 追加モデル2
    "XGBoost": XGBRegressor(random_state=42),
    "LightGBM": LGBMRegressor(random_state=42, verbose=-1),
}

# 2. クロスバリデーション(K=5)とRFEの設定
# ※エラーと速度低下を防ぐため、RFEの基準モデルにはLinearRegressionを固定で使用（特徴量を半数に削減）
kf = KFold(n_splits=5, shuffle=True, random_state=42)
rfe = RFE(estimator=LinearRegression())

# 3. for文で各モデルをループして評価
results = {}
for name, model in models.items():
    # データリークを防ぐため、RFEとモデルをパイプラインで結合
    pipe = Pipeline([('rfe', rfe), ('model', model)])

    # 交差検証の実行（評価指標はRMSEを使用）
    scores = cross_val_score(pipe, X_all_train, y_all_train, cv=kf, scoring='neg_mean_squared_error', n_jobs=-1)
    rmse = np.sqrt(-scores.mean())

    results[name] = rmse
    print(f"{name}: RMSE = {rmse:.4f}")

# 4. 最も精度の良い（RMSEが最も低い）モデルの出力
best_model = min(results, key=results.get)
print(f"\n★ 最も精度が良いモデル: {best_model} (RMSE: {results[best_model]:.4f})")

LinearRegression: RMSE = 0.7342
Ridge: RMSE = 0.7342
Lasso: RMSE = 0.9877
ElasticNet: RMSE = 0.8970
RandomForest: RMSE = 0.4929
GradientBoosting: RMSE = 0.5696
XGBoost: RMSE = 0.4938
LightGBM: RMSE = 0.4912

★ 最も精度が良いモデル: LightGBM (RMSE: 0.4912)


In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_selection import RFE  # RFE（特徴量削減）をインポート
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold

train = pd.read_csv(
    "/content/drive/MyDrive/house_price_competition_hoshino/data/train_data_raw.csv"
)
test = pd.read_csv(
    "/content/drive/MyDrive/house_price_competition_hoshino/data/test_data_raw.csv"
)
sample = pd.read_csv(
    "/content/drive/MyDrive/house_price_competition_hoshino/data/sample_raw.csv"
)

# 2. データの切り分け
X_all_train = train.drop(["Price", "id"], axis=1)
y_all_train = train["Price"]
X_test = test.drop(["id"], axis=1) if "id" in test.columns else test

print("データ分割前の行数:", len(X_all_train))

# 3. クロスバリデーションおよびRFEの設定
folds_num = 5
kf = KFold(n_splits=folds_num, shuffle=True, random_state=42)

# 各種予測値やスコアを格納する配列
oof_predictions = np.zeros(len(X_all_train))
test_predictions = np.zeros(len(X_test))
cv_scores = []

# 4. クロスバリデーションのループ処理（RFEを含む）
for fold, (cv_train_idx, cv_val_idx) in enumerate(
    kf.split(X_all_train, y_all_train)
):
    print(f"--- Fold {fold + 1} / {folds_num} 開始 ---")

    # 1. このフォールドの訓練データと検証データに分割
    X_cv_train, y_cv_train = (
        X_all_train.iloc[cv_train_idx],
        y_all_train.iloc[cv_train_idx],
    )
    X_cv_val, y_cv_val = (
        X_all_train.iloc[cv_val_idx],
        y_all_train.iloc[cv_val_idx],
    )

    # 2. RFE（特徴量選択）の実行
    print(f"Fold {fold + 1}: RFEによる特徴量選択中...")
    rfe_base_model = LGBMRegressor(random_state=42, verbose=-1)
    rfe_selector = RFE(
        estimator=rfe_base_model,
        step=1,
    )

    # このフォールドの訓練データのみを使って、どの特徴量を残すか学習
    rfe_selector.fit(X_cv_train, y_cv_train)

    # 3. 選択された「無駄のない特徴量」だけにデータを絞り込む
    X_cv_train_selected = rfe_selector.transform(X_cv_train)
    X_cv_val_selected = rfe_selector.transform(X_cv_val)
    X_test_selected = rfe_selector.transform(X_test)

    selected_features = X_all_train.columns[rfe_selector.support_]
    print(f"選ばれた特徴量: {list(selected_features)}")

    # 4. 絞り込んだ特徴量を使って、最終的な予測モデルを学習
    model = LGBMRegressor(random_state=42, verbose=-1)
    model.fit(X_cv_train_selected, y_cv_train)

    # 5. 検証データおよびテストデータに対する予測
    val_preds = model.predict(X_cv_val_selected)

    # 【追加】検証データの数値の丸め込み処理
    val_preds = np.where(val_preds <= 0, 0, val_preds)
    val_preds = np.where(val_preds >= 5.00001, 5.00001, val_preds)

    oof_predictions[cv_val_idx] = val_preds

    # 各フォールドで絞り込まれたテストデータを使って予測し、平均化
    test_preds = model.predict(X_test_selected)

    # 【追加】テストデータの数値の丸め込み処理
    test_preds = np.where(test_preds <= 0, 0, test_preds)
    test_preds = np.where(test_preds >= 5.00001, 5.00001, test_preds)

    test_predictions += test_preds / folds_num

    # 6. このフォールドのスコアを記録
    fold_rmse = np.sqrt(mean_squared_error(y_cv_val, val_preds))
    cv_scores.append(fold_rmse)
    print(f"Fold {fold + 1} の検証スコア (RMSE): {fold_rmse:.4f}")

# 5. クロスバリデーション全体の評価スコア算出
print("\n==================================")
print(f"各Foldの平均スコア: {np.mean(cv_scores):.4f}")
overall_oof_score = np.sqrt(mean_squared_error(y_all_train, oof_predictions))
print(f"全体のOOFスコア (RMSE): {overall_oof_score:.4f}")
print("==================================")

# 6. 提出用ファイルの作成
sample["Price"] = test_predictions

# CSVとして出力（必要に応じてコメントアウトを解除してください）
# sample.to_csv('/content/drive/MyDrive/house_price_competition_hoshino/data/submission_cv_rfe.csv', index=False)
# print("提出用ファイルを保存しました。")

データ分割前の行数: 16512
--- Fold 1 / 5 開始 ---
Fold 1: RFEによる特徴量選択中...
選ばれた特徴量: ['MedInc', 'AveRooms', 'AveOccup', 'Latitude', 'Longitude']
Fold 1 の検証スコア (RMSE): 0.4908
--- Fold 2 / 5 開始 ---


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Fold 2: RFEによる特徴量選択中...
選ばれた特徴量: ['MedInc', 'AveRooms', 'AveOccup', 'Latitude', 'Longitude']
Fold 2 の検証スコア (RMSE): 0.4723
--- Fold 3 / 5 開始 ---


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Fold 3: RFEによる特徴量選択中...
選ばれた特徴量: ['MedInc', 'AveRooms', 'AveOccup', 'Latitude', 'Longitude']


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Fold 3 の検証スコア (RMSE): 0.4831
--- Fold 4 / 5 開始 ---
Fold 4: RFEによる特徴量選択中...
選ばれた特徴量: ['MedInc', 'AveRooms', 'AveOccup', 'Latitude', 'Longitude']


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Fold 4 の検証スコア (RMSE): 0.4605
--- Fold 5 / 5 開始 ---
Fold 5: RFEによる特徴量選択中...
選ばれた特徴量: ['MedInc', 'AveRooms', 'AveOccup', 'Latitude', 'Longitude']
Fold 5 の検証スコア (RMSE): 0.4752

各Foldの平均スコア: 0.4764
全体のOOFスコア (RMSE): 0.4765


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [ ]:
# csvファイルの作成
sample.to_csv('submit_raw_cv_lgbm.csv',index=None)

In [ ]:
# 作成したファイルをKaggleに直接投稿
# -c:提出コンペ名
# -f：ファイル名
# -m：投稿コメント

!kaggle competitions submit -c ambl-california-housing -f submit_raw_cv_lgbm.csv -m "Yeah! I submit my file through the Google Colab!"

100% 94.0k/94.0k [00:00<00:00, 128kB/s]
Successfully submitted to AMBL初心者向けコンペティション_California Housing